In [257]:
import json
import re
from yarn_utils import YARNGraph

In [298]:
FOLDER_PATH = "annotations/FRACAS_12032026/"
FILE = "107h.yarn.json"

In [299]:
with open(FOLDER_PATH + FILE) as f:
    yarn_graph_json = json.load(f)

In [300]:
yarn_graph = YARNGraph(yarn_graph_json)

In [301]:
output = yarn_graph.grew()

In [302]:
with open('output.json', 'w') as f:
    json.dump(output, f)

In [303]:
# Create R by getting all relations between V nodes

relations = set()
for node, feats in output['nodes'].items():
    if feats['type'] == "E":
        edge_label = feats['rel']

        for edge in output['edges']:
            if edge['tar'] == node:
                src = edge['src']
                
            if edge['src'] == node:
                tar = edge['tar']
                

        relations.add((edge_label, src, tar))

R = {}
for relation in relations:
    label = relation[0]
    src = relation[1]
    tar = relation[2]

    if src not in R:
        R[src] = [(label, src, tar)]
    else:
        R[src].append((label, src, tar))

In [304]:
variables = set()
F = set()

for node, feats in output['nodes'].items():
    if feats['type'] == "L" or feats['type'] == 'H':
        for edge in output['edges']:
            if edge['src'] == node:
                tar = edge['tar']

                if output['nodes'][tar]['type'] == 'V':
                    edge_label = feats['value']
                    if 'pred' in output['nodes'][tar]:
                        tar_label = output['nodes'][tar]['pred']
                    else:
                        tar_label = output['nodes'][tar]['concept']
        
                    base = tar_label[0]
                    i = 0

                    while True:
                        variable = base if i == 0 else f"{base}{i}"
                        if variable not in variables:
                            variables.add(variable)
                            break
                        i += 1

                    F.add((("Q_"+edge_label, variable, tar_label), tar, node))
    
    if feats['type'] == 'H':
        for edge in output['edges']:
            if edge['src'] == node:
                tar = edge['tar']

                if output['nodes'][tar]['type'] == 'L' or output['nodes'][tar]['type'] == 'H':
                    edge_label = feats['value']

                    F.add((("Q_"+edge_label), tar, node))

# Ignore definitness and number for now
F = {f for f in F if f[0][0] not in ['Q_definite', 'Q_indefinite', 'Q_singular', 'Q_plural']}


In [305]:
F

{(('Q_exists', 'a1', 'accountant'), 'va1', 'l2'),
 (('Q_exists', 'm', 'meeting'), 'vm1', 'l3'),
 (('Q_past', 'a', 'attend-01'), 'va2', 'l1')}

In [306]:
R

{'va2': [('ARG0', 'va2', 'va1'), ('ARG1', 'va2', 'vm1')]}

In [ ]:
forest = []

for f in F:
    if isinstance(f[0], tuple):
        forest.append({
            'src':f[2],
            'tar':f[1],
            'type': f[0][0],
            'variable': f[0][1],
            'concept': f[0][2],
            'children': None,
            'relations': [f"{rel[0]}({rel[1]},{rel[2]})" for rel in R[f[1]]] if f[1] in R else []
        })
    else:
        forest.append({
            'src':f[2],
            'tar':f[1],
            'type': f[0][0],
            'variable': None,
            'concept': None,
            'children': f[1],
            'relations': [f"{rel[0]}({rel[1]},{rel[2]})" for rel in R[f[1]]] if f[1] in R else []
        })

# C
to_delete = []
for src, rels in R.items():
    for rel in rels:
        tar = rel[2]

        for f1 in forest:
            if f1['tar'] == tar:
                for f2 in forest:
                    if f2['tar'] == src:
                        f1['children'] = f2
                        to_delete.append(f2)

# for f1 in forest:
#     if isinstance(f1['children'], str):
#         for f2 in forest:
#             if f2['src'] == f1['children']:
#                 f1['children'] = f2
#                 print(f2)
#                 to_delete.append(f2)

forest = [f for f in forest if f not in to_delete]

In [308]:
forest

[{'src': 'l2',
  'tar': 'va1',
  'type': 'Q_exists',
  'variable': 'a1',
  'concept': 'accountant',
  'children': {'src': 'l1',
   'tar': 'va2',
   'type': 'Q_past',
   'variable': 'a',
   'concept': 'attend-01',
   'children': None,
   'relations': ['ARG0(va2,va1)', 'ARG1(va2,vm1)']},
  'relations': []},
 {'src': 'l3',
  'tar': 'vm1',
  'type': 'Q_exists',
  'variable': 'm',
  'concept': 'meeting',
  'children': {'src': 'l1',
   'tar': 'va2',
   'type': 'Q_past',
   'variable': 'a',
   'concept': 'attend-01',
   'children': None,
   'relations': ['ARG0(va2,va1)', 'ARG1(va2,vm1)']},
  'relations': []}]

In [309]:
T_all = []
for i in forest:
    children_i = i['children']
    for j in forest:
        children_j = j['children']
        if i != j and children_i == children_j:
            i_top = i.copy()
            j_top = j.copy()
            i_top['children'] = j
            j_top['children'] = i
            T_all.append(i_top)
            T_all.append(j_top)

In [310]:
T_all

[{'src': 'l2',
  'tar': 'va1',
  'type': 'Q_exists',
  'variable': 'a1',
  'concept': 'accountant',
  'children': {'src': 'l3',
   'tar': 'vm1',
   'type': 'Q_exists',
   'variable': 'm',
   'concept': 'meeting',
   'children': {'src': 'l1',
    'tar': 'va2',
    'type': 'Q_past',
    'variable': 'a',
    'concept': 'attend-01',
    'children': None,
    'relations': ['ARG0(va2,va1)', 'ARG1(va2,vm1)']},
   'relations': []},
  'relations': []},
 {'src': 'l3',
  'tar': 'vm1',
  'type': 'Q_exists',
  'variable': 'm',
  'concept': 'meeting',
  'children': {'src': 'l2',
   'tar': 'va1',
   'type': 'Q_exists',
   'variable': 'a1',
   'concept': 'accountant',
   'children': {'src': 'l1',
    'tar': 'va2',
    'type': 'Q_past',
    'variable': 'a',
    'concept': 'attend-01',
    'children': None,
    'relations': ['ARG0(va2,va1)', 'ARG1(va2,vm1)']},
   'relations': []},
  'relations': []},
 {'src': 'l3',
  'tar': 'vm1',
  'type': 'Q_exists',
  'variable': 'm',
  'concept': 'meeting',
  'child

In [311]:
T_all[0]

{'src': 'l2',
 'tar': 'va1',
 'type': 'Q_exists',
 'variable': 'a1',
 'concept': 'accountant',
 'children': {'src': 'l3',
  'tar': 'vm1',
  'type': 'Q_exists',
  'variable': 'm',
  'concept': 'meeting',
  'children': {'src': 'l1',
   'tar': 'va2',
   'type': 'Q_past',
   'variable': 'a',
   'concept': 'attend-01',
   'children': None,
   'relations': ['ARG0(va2,va1)', 'ARG1(va2,vm1)']},
  'relations': []},
 'relations': []}

In [ ]:
def interpret(root, temp_variable, colors = False):
     if root == None:
         return ""
     if root["type"] == "Q_exists":
         return "∃" + root["variable"] + ". (" + root["concept"] + "(" + root['variable'] + ") ∧ " + \
     " ∧ ".join(root['relations']) + "(" + interpret(root['children'], temp_variable) + ")"
     if root["type"] == "Q_past":
         return "∃" + root["variable"] + ". (" + root["concept"] + "(" + root['variable'] + ") ∧ " + \
     root['variable'] + "≺" + temp_variable + " ∧ " + \
     " ∧ ".join(root['relations']) + "(" + interpret(root['children'], root['variable']) + ")"
     
def clean_formula(formula):
    return formula.replace("()", "")     


In [315]:
print(clean_formula(interpret(T_all[0], "now")))
print(clean_formula(interpret(T_all[1], "now")))

∃a1. (accountant(a1) ∧ (∃m. (meeting(m) ∧ (∃a. (attend-01(a) ∧ a≺now ∧ ARG0(va2,va1) ∧ ARG1(va2,vm1)))
∃m. (meeting(m) ∧ (∃a1. (accountant(a1) ∧ (∃a. (attend-01(a) ∧ a≺now ∧ ARG0(va2,va1) ∧ ARG1(va2,vm1)))
